In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [13]:
!pip install streamlit groq pandas chromadb sentence-transformers -q

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from groq import Groq
import chromadb
from sentence_transformers import SentenceTransformer
import hashlib

st.title("🧙 IA com RAG - Gandalf")
st.caption("Busca vetorial — aguenta qualquer tamanho")

api_key = st.text_input("API Key do Groq:", type="password")

# Modelos (carrega uma vez)
@st.cache_resource
def carregar_modelos():
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    chroma_client = chromadb.Client()
    return embedding_model, chroma_client

embedding_model, chroma_client = carregar_modelos()

uploaded_files = st.file_uploader(
    "📂 Envie seus CSVs ou Excel",
    type=["csv", "xlsx", "xls"],
    accept_multiple_files=True
)

def indexar_dataframe(df, nome_arquivo, collection):
    """Divide o CSV em chunks e indexa no ChromaDB"""
    chunk_size = 50  # linhas por chunk
    chunks = []
    ids = []

    for i in range(0, len(df), chunk_size):
        chunk = df.iloc[i:i+chunk_size]
        texto = f"Arquivo: {nome_arquivo}\nLinhas {i} a {i+len(chunk)}:\n{chunk.to_string()}"
        chunk_id = hashlib.md5(texto.encode()).hexdigest()
        chunks.append(texto)
        ids.append(chunk_id)

    # Gera embeddings e salva
    embeddings = embedding_model.encode(chunks).tolist()
    collection.add(documents=chunks, embeddings=embeddings, ids=ids)
    return len(chunks)

if uploaded_files:
    # Cria ou recupera collection
    collection_name = "datasets"
    try:
        collection = chroma_client.get_collection(collection_name)
    except:
        collection = chroma_client.create_collection(collection_name)

    for file in uploaded_files:
        file_hash = hashlib.md5(file.name.encode()).hexdigest()

        if f"indexed_{file_hash}" not in st.session_state:
            with st.spinner(f"Indexando {file.name}..."):
                if file.name.endswith(".csv"):
                    df = pd.read_csv(file)
                else:
                    df = pd.read_excel(file)

                n_chunks = indexar_dataframe(df, file.name, collection)
                st.session_state[f"indexed_{file_hash}"] = True
                st.success(f"✅ {file.name} indexado em {n_chunks} blocos ({df.shape[0]} linhas)")
        else:
            st.info(f"✅ {file.name} já está indexado")

    st.divider()

    if "messages" not in st.session_state:
        st.session_state.messages = []

    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.write(msg["content"])

    if prompt := st.chat_input("olá sou Gandalf, seu Mentor do Dinheiro, como posso te ajudar hoje?"):
        if not api_key:
            st.error("Insira sua API key primeiro!")
        else:
            st.session_state.messages.append({"role": "user", "content": prompt})
            with st.chat_message("user"):
                st.write(prompt)

            # Busca os chunks mais relevantes
            query_embedding = embedding_model.encode([prompt]).tolist()
            resultados = collection.query(
                query_embeddings=query_embedding,
                n_results=5  # pega os 5 pedaços mais relevantes
            )
            contexto = "\n\n".join(resultados["documents"][0])

            system_prompt = f"""Você é um analista de dados experiente.
Responda a pergunta do usuário com base nos trechos do dataset abaixo.
Se não encontrar a informação, Isto esta além da minha compreensão.

TRECHOS RELEVANTES:
{contexto}
"""
            client = Groq(api_key=api_key)
            mensagens = [
                {"role": "system", "content": system_prompt},
                *st.session_state.messages
            ]

            with st.chat_message("assistant"):
                with st.spinner("Entendi pequeno mestre,vou verificar para você"):
                    response = client.chat.completions.create(
                        model="llama-3.3-70b-versatile",
                        messages=mensagens,
                        max_tokens=2048
                    )
                    reply = response.choices[0].message.content
                    st.write(reply)

            st.session_state.messages.append({"role": "assistant", "content": reply})

else:
    st.info("⬆️ Envie seus arquivos para começar")

In [ ]:

import subprocess, threading, time, re

def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py",
                    "--server.port=8501",
                    "--server.headless=true"])

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(4)

# Inicia o túnel Cloudflare
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stderr=subprocess.PIPE, stdout=subprocess.PIPE
)

# Captura a URL gerada
for line in tunnel.stderr:
    line = line.decode()
    if "trycloudflare.com" in line:
        url = re.search(r'https://\S+\.trycloudflare\.com', line)
        if url:
            print(f"✅ Acesse aqui: {url.group()}")
            break